### 0. Setup and test

In [30]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jeanmidev/smart-meters-in-london")

print("Path to dataset files:", path)

c:\Users\2553227\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\2553227\.cache\kagglehub\datasets\jeanmidev\smart-meters-in-london\versions\21


In [31]:
import pandas as pd
import numpy as np

from pathlib import Path

# Set Data directory
DATA_DIR = Path("data")

# For reproducibility
RNG_SEED = 42
np.random.seed(RNG_SEED)

### 1. Inspect available files

In [32]:
list(DATA_DIR.glob("**/*.csv"))

[WindowsPath('data/acorn_details.csv'),
 WindowsPath('data/daily_dataset.csv'),
 WindowsPath('data/informations_households.csv'),
 WindowsPath('data/uk_bank_holidays.csv'),
 WindowsPath('data/weather_daily_darksky.csv'),
 WindowsPath('data/weather_hourly_darksky.csv'),
 WindowsPath('data/daily_dataset/block_0.csv'),
 WindowsPath('data/daily_dataset/block_1.csv'),
 WindowsPath('data/daily_dataset/block_10.csv'),
 WindowsPath('data/daily_dataset/block_100.csv'),
 WindowsPath('data/daily_dataset/block_101.csv'),
 WindowsPath('data/daily_dataset/block_102.csv'),
 WindowsPath('data/daily_dataset/block_103.csv'),
 WindowsPath('data/daily_dataset/block_104.csv'),
 WindowsPath('data/daily_dataset/block_105.csv'),
 WindowsPath('data/daily_dataset/block_106.csv'),
 WindowsPath('data/daily_dataset/block_107.csv'),
 WindowsPath('data/daily_dataset/block_108.csv'),
 WindowsPath('data/daily_dataset/block_109.csv'),
 WindowsPath('data/daily_dataset/block_11.csv'),
 WindowsPath('data/daily_dataset/blo

### 2. Load metadata (households + ACORN + tariff)

In [33]:
# acorn_path = DATA_DIR / "acorn_details.csv"
info_path  = DATA_DIR / "informations_households.csv"

# acorn_df = pd.read_csv(acorn_path, encoding="latin1")
info_df  = pd.read_csv(info_path, encoding="latin1")

# print(acorn_df.head())
print(info_df.head())

       LCLid stdorToU    Acorn Acorn_grouped     file
0  MAC005492      ToU   ACORN-        ACORN-  block_0
1  MAC001074      ToU   ACORN-        ACORN-  block_0
2  MAC000002      Std  ACORN-A      Affluent  block_0
3  MAC003613      Std  ACORN-A      Affluent  block_0
4  MAC003597      Std  ACORN-A      Affluent  block_0


In [34]:
# meta = (
#     acorn_df.merge(info_df, on="LCLid", how="inner")
# )

meta = info_df.copy()

meta.head()
# meta["stdorToU"].value_counts()
# meta["Acorn_grouped"].value_counts()

,LCLid,stdorToU,Acorn,Acorn_grouped,file
0,MAC005492,ToU,ACORN-,ACORN-,block_0
1,MAC001074,ToU,ACORN-,ACORN-,block_0
2,MAC000002,Std,ACORN-A,Affluent,block_0
3,MAC003613,Std,ACORN-A,Affluent,block_0
4,MAC003597,Std,ACORN-A,Affluent,block_0


### 3. Sample households (balanced by tariff & ACORN_grouped)

In [35]:
# Keep only columns you need for stratification
meta = info_df[["LCLid", "stdorToU", "Acorn_grouped"]].dropna()

# Create a combined stratum label
meta["stratum"] = (
    meta["stdorToU"].astype(str) + "_" +
    meta["Acorn_grouped"].astype(str)
)

# Decide how many households per stratum (e.g. up to 50)
n_per_stratum = 50

sampled_ids = (
    meta
    .groupby("stratum", group_keys=False)
    .apply(lambda g: g.sample(
        n=min(len(g), n_per_stratum),
        random_state=42
    ))
    ["LCLid"]
    .unique()
)

len(sampled_ids), sampled_ids[:10]
sampled_meta = info_df[info_df["LCLid"].isin(sampled_ids)].copy()
sampled_meta["file"].value_counts()

file
block_110    34
block_111    16
block_41      9
block_6       8
block_51      8
             ..
block_76      1
block_90      1
block_95      1
block_96      1
block_101     1
Name: count, Length: 103, dtype: int64

### 4. Find households' daily data

In [36]:
blocks_to_load = sorted(sampled_meta["file"].unique())
blocks_to_load[:10], len(blocks_to_load)

dfs = []
for block in blocks_to_load:
    path = DATA_DIR / f"daily_dataset/{block}.csv"
    tmp = pd.read_csv(path)
    tmp = tmp[tmp["LCLid"].isin(sampled_ids)]
    dfs.append(tmp)

daily = pd.concat(dfs, ignore_index=True)

# Ensure proper types and ordering
daily["day"] = pd.to_datetime(daily["day"])
daily = daily.sort_values(["LCLid", "day"])

# Count number of days per household
day_counts = (
    daily.groupby("LCLid")["day"]
         .nunique()
         .rename("n_days")
)

print(day_counts.describe())

# Keep households with enough days (e.g. at least 200)
MIN_DAYS = 200
good_ids = day_counts[day_counts >= MIN_DAYS].index

# Restrict to good households
daily = daily[daily["LCLid"].isin(good_ids)].copy()
sampled_meta = sampled_meta[sampled_meta["LCLid"].isin(good_ids)].copy()

count    351.000000
mean     620.062678
std      121.967919
min        3.000000
25%      595.500000
50%      645.000000
75%      681.000000
max      828.000000
Name: n_days, dtype: float64


### 5. Pick target

In [37]:
# Predict same-day energy_mean
daily_use = daily[["LCLid", "day", "energy_mean"]].copy()

### 6. Joing metadata

In [38]:
# Merge metadata: tariff and Acorn are the features of interest
model_df = daily_use.merge(
    sampled_meta[["LCLid", "stdorToU", "Acorn_grouped"]],
    on="LCLid",
    how="left"
)

print(model_df.head())
print(model_df.shape)

       LCLid        day  energy_mean stdorToU Acorn_grouped
0  MAC000023 2011-12-07     0.431778      Std       ACORN-U
1  MAC000023 2011-12-08     0.448375      Std       ACORN-U
2  MAC000023 2011-12-09     0.303417      Std       ACORN-U
3  MAC000023 2011-12-10     0.229833      Std       ACORN-U
4  MAC000023 2011-12-11     0.391438      Std       ACORN-U
(217137, 5)


### 7. Add context

In [39]:
# Load weather
weather_path = DATA_DIR / "weather_daily_darksky.csv"
weather = pd.read_csv(weather_path)

# Inspect columns
print(weather.head())
print(weather.columns)

# Parse date and keep useful columns
# Adjust column names if needed after print(...)
weather["day"] = pd.to_datetime(weather["time"]).dt.date
weather["day"] = pd.to_datetime(weather["day"])

weather_daily = (
    weather
    .groupby("day")
    .agg({
        "temperatureMin": "mean",
        "temperatureMax": "mean",
        "humidity":       "mean",
        "windSpeed":      "mean",
        "cloudCover":     "mean",
    })
    .reset_index()
)

   temperatureMax   temperatureMaxTime  windBearing                 icon  \
0           11.96  2011-11-11 23:00:00          123                  fog   
1            8.59  2011-12-11 14:00:00          198    partly-cloudy-day   
2           10.33  2011-12-27 02:00:00          225    partly-cloudy-day   
3            8.07  2011-12-02 23:00:00          232                 wind   
4            8.22  2011-12-24 23:00:00          252  partly-cloudy-night   

   dewPoint   temperatureMinTime  cloudCover  windSpeed  pressure  \
0      9.40  2011-11-11 07:00:00        0.79       3.88   1016.08   
1      4.49  2011-12-11 01:00:00        0.56       3.94   1007.71   
2      5.47  2011-12-27 23:00:00        0.85       3.54   1032.76   
3      3.69  2011-12-02 07:00:00        0.32       3.00   1012.12   
4      2.79  2011-12-24 07:00:00        0.37       4.46   1028.17   

  apparentTemperatureMinTime  ...  temperatureHigh          sunriseTime  \
0        2011-11-11 07:00:00  ...            10.87  2

In [41]:
# daily currently has LCLid, day, energy_mean, ...
model_df = model_df.merge(weather_daily, on="day", how="left")

In [47]:
model_df = model_df.sort_values("day").reset_index(drop=True)

weather_cols = ["temperatureMin", "temperatureMax", "humidity", "windSpeed", "cloudCover"]
model_df[weather_cols] = model_df[weather_cols].ffill().bfill()

model_df["is_bank_holiday"] = model_df["is_bank_holiday"].fillna(False)

# Confirm
print(model_df.isnull().sum())

LCLid                        0
day                          0
energy_mean                  0
dayofweek                    0
month                        0
season                       0
stdorToU_ToU                 0
Acorn_grouped_ACORN-U        0
Acorn_grouped_Adversity      0
Acorn_grouped_Affluent       0
Acorn_grouped_Comfortable    0
temperatureMin               0
temperatureMax               0
humidity                     0
windSpeed                    0
cloudCover                   0
is_bank_holiday              0
dtype: int64


In [40]:
# Calendar context features
model_df["dayofweek"] = model_df["day"].dt.dayofweek
model_df["month"]     = model_df["day"].dt.month
model_df["season"]    = model_df["month"].map(
    {12:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2, 9:3,10:3,11:3}
)

# Target = energy_mean (no shift)
model_df = model_df.dropna(subset=["energy_mean"])

# One-hot encode small categoricals (good for both ML and simple DL MLPs)
model_df = pd.get_dummies(
    model_df,
    columns=["stdorToU", "Acorn_grouped"],
    drop_first=True
)

print(model_df.head())
print(model_df.shape)

       LCLid        day  energy_mean  dayofweek  month  season  stdorToU_ToU  \
0  MAC000023 2011-12-07     0.431778          2     12       0         False   
1  MAC000023 2011-12-08     0.448375          3     12       0         False   
2  MAC000023 2011-12-09     0.303417          4     12       0         False   
3  MAC000023 2011-12-10     0.229833          5     12       0         False   
4  MAC000023 2011-12-11     0.391438          6     12       0         False   

   Acorn_grouped_ACORN-U  Acorn_grouped_Adversity  Acorn_grouped_Affluent  \
0                   True                    False                   False   
1                   True                    False                   False   
2                   True                    False                   False   
3                   True                    False                   False   
4                   True                    False                   False   

   Acorn_grouped_Comfortable  
0                      Fa

In [26]:
# Load UK bank holidays
hol_path = DATA_DIR / "uk_bank_holidays.csv"
hol = pd.read_csv(hol_path)

# Ensure date column matches your naming; assume 'date' for now
hol["day"] = pd.to_datetime(hol["Bank holidays"])
hol["is_bank_holiday"] = True

holidays = hol[["day", "is_bank_holiday"]].drop_duplicates()

In [42]:
model_df = model_df.merge(holidays, on="day", how="left")
model_df["is_bank_holiday"] = model_df["is_bank_holiday"].fillna(False)

### 8. Save dataset

In [48]:
dataset_path = "smart_meters_daily_supervised.csv"
model_df.to_csv(dataset_path, index=False)

dataset_path

'smart_meters_daily_supervised.csv'

### Extra: test model for meaningful results

In [50]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score
import pandas as pd
import numpy as np
from pathlib import Path

dataset_path = "smart_meters_daily_supervised.csv"
model_df = pd.read_csv(dataset_path)
model_df["day"] = pd.to_datetime(model_df["day"])

# Target
y = model_df["energy_mean"]

# Features (all except ID, day, target)
feature_cols = [
    "dayofweek",
    "month",
    "season",
    "stdorToU_ToU",
    "Acorn_grouped_ACORN-U",
    "Acorn_grouped_Adversity",
    "Acorn_grouped_Affluent",
    "Acorn_grouped_Comfortable",
    "temperatureMin",
    "temperatureMax",
    "humidity",
    "windSpeed",
    "cloudCover",
    "is_bank_holiday",
]

# Time-based split: last 20% of days as validation
cutoff_date = model_df["day"].quantile(0.8)

train_mask = model_df["day"] <= cutoff_date
val_mask   = model_df["day"] >  cutoff_date

X_train = model_df.loc[train_mask, feature_cols]
y_train = y[train_mask]

X_val   = model_df.loc[val_mask, feature_cols]
y_val   = y[val_mask]

In [51]:
lin = LinearRegression()
lin.fit(X_train, y_train)

y_pred = lin.predict(X_val)

rmse = root_mean_squared_error(y_val, y_pred)
r2   = r2_score(y_val, y_pred)

rmse, r2

(0.18182853464977006, 0.012249214170635625)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
import pandas as pd
import numpy as np
from pathlib import Path
RNG_SEED = 42
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=RNG_SEED,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)

rmse_rf = root_mean_squared_error(y_val, y_pred_rf)
r2_rf   = r2_score(y_val, y_pred_rf)

rmse_rf, r2_rf

(0.20790203273700655, -0.2913405167263592)